Cell 1: Environment Setup

In [ ]:
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install -q torch-pruning networkx codecarbon thop psutil

Cell 2: Imports and Reproducibility

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import random
import os
import pandas as pd
import matplotlib.pyplot as plt

from codecarbon import EmissionsTracker
from thop import profile
import torch_pruning as tp


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Cell 3: Data Preparation

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# Smaller dataset for fast training
train_subset = Subset(trainset, range(10000))
trainloader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

Cell 4: Utility Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        _, pred = out.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()
    return 100.0 * correct / total


def train_kd_epoch(model, teacher, loader, optimizer, T=3.0, alpha=0.5):
    model.train()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            teacher_logits = teacher(x)
        student_logits = model(x)

        # KD Loss
        soft_loss = nn.KLDivLoss(reduction="batchmean")(
            torch.log_softmax(student_logits / T, dim=1),
            torch.softmax(teacher_logits / T, dim=1),
        ) * (T * T)
        hard_loss = nn.CrossEntropyLoss()(student_logits, y)

        loss = alpha * soft_loss + (1.0 - alpha) * hard_loss
        loss.backward()
        optimizer.step()

        # Calculate accuracy
        _, pred = student_logits.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()

    return 100.0 * correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return 100.0 * correct / total


def model_size(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def latency_bs1(model, runs=50):
    model.eval()
    x = torch.randn(1, 3, 224, 224).to(device)
    times = []
    with torch.no_grad():
        for _ in range(runs):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.time()
            model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - start)
    return np.mean(times) * 1000

Cell 5: Baseline Model Training

In [5]:
model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(512, 10)
model = model.to(device)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("\n=== FAST TRANSFER TRAINING (5 epochs) ===")
tracker = EmissionsTracker(log_level="error")
tracker.start()

for epoch in range(5):
    acc = train_epoch(model, trainloader, optimizer, criterion)
    print(f"Epoch {epoch + 1}/5 | Train Acc {acc:.2f}%")

baseline_acc = evaluate(model, testloader)
baseline_emissions = tracker.stop()

baseline_flops, _ = profile(
    model, inputs=(torch.randn(1, 3, 224, 224).to(device),), verbose=False
)
baseline_params = model_size(model)
baseline_latency = latency_bs1(model)

print(
    f"\nBASELINE RESULTS\nAccuracy: {baseline_acc}\nParams(M): {baseline_params}\nFLOPs(G): {baseline_flops / 1e9}\nLatency: {baseline_latency}ms"
)

Cell 6: Structured Pruning

In [ ]:
print("\n=== FAST STRUCTURED PRUNING ===")
model.eval()
example_inputs = torch.randn(1, 3, 224, 224).to(device)

for param in model.parameters():
    param.requires_grad = True

imp = tp.importance.MagnitudeImportance(p=2)
pruner = tp.pruner.MetaPruner(
    model,
    example_inputs,
    importance=imp,
    pruning_ratio=0.35,
    iterative_steps=1,
    ignored_layers=[model.fc],
)
pruner.step()
print("Pruning completed")

In [ ]:
print("\n=== KNOWLEDGE DISTILLATION SETUP ===")

# Add Teacher Model
teacher_model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
teacher_model.fc = nn.Linear(512, 10)  # Match your classes
teacher_model.to(device).eval()

print("Teacher model loaded and KD training function defined")

In [ ]:
for p in model.parameters():
    p.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=3e-4)

print("\nFine-tuning pruned model with Knowledge Distillation (3 epochs)")
tracker = EmissionsTracker(log_level="error")
tracker.start()

for epoch in range(3):
    acc = train_kd_epoch(model, teacher_model, trainloader, optimizer, T=3.0, alpha=0.5)
    print(f"Epoch {epoch + 1}/3 | Train Acc {acc:.2f}% (KD)")

pruned_acc = evaluate(model, testloader)
pruned_emissions = tracker.stop()
pruned_flops, _ = profile(
    model, inputs=(torch.randn(1, 3, 224, 224).to(device),), verbose=False
)
pruned_params = model_size(model)
pruned_latency = latency_bs1(model)

# RESULTS TABLE
print(
    "\n" + "=" * 70 + "\nFINAL RESULTS TABLE (WITH KNOWLEDGE DISTILLATION)\n" + "=" * 70
)
print(f"{'Metric':<20}{'Baseline':<15}{'Pruned+KD':<15}{'Change'}")
print("-" * 70)
print(
    f"{'Accuracy':<20}{baseline_acc:<15.2f}{pruned_acc:<15.2f}{pruned_acc - baseline_acc:+.2f}"
)
print(
    f"{'Params(M)':<20}{baseline_params:<15.2f}{pruned_params:<15.2f}{pruned_params / baseline_params * 100 - 100:+.1f}%"
)
print(
    f"{'FLOPs(G)':<20}{baseline_flops / 1e9:<15.2f}{pruned_flops / 1e9:<15.2f}{pruned_flops / baseline_flops * 100 - 100:+.1f}%"
)
print(
    f"{'Latency(ms)':<20}{baseline_latency:<15.2f}{pruned_latency:<15.2f}{pruned_latency / baseline_latency * 100 - 100:+.1f}%"
)
print(
    f"{'CO2(g)':<20}{baseline_emissions:<15.4f}{pruned_emissions:<15.4f}{pruned_emissions / baseline_emissions * 100 - 100:+.1f}%"
)
print("=" * 70)

Cell 8: Export

In [ ]:
print("\n=== EXPORTING TO TORCHSCRIPT ===")
model.cpu()
scripted = torch.jit.script(model)
os.makedirs("outputs", exist_ok=True)
scripted.save("outputs/scripted_pruned_resnet18.pt")
torch.save(model, "outputs/fast_pruned_resnet18.pth")
print("Model saved to 'outputs/' directory.")

Testing

In [ ]:
# Prepare experiment metrics
data = {
    "Metric": ["Accuracy (%)", "Params (M)", "FLOPs (G)", "Latency (ms)"],
    "Baseline": [
        baseline_acc,
        baseline_params,
        baseline_flops / 1e9,
        float(baseline_latency),
    ],
    "Pruned": [pruned_acc, pruned_params, pruned_flops / 1e9, float(pruned_latency)],
}

df_metrics = pd.DataFrame(data)

# Percentage change
df_metrics["Change (%)"] = (
    (df_metrics["Pruned"] - df_metrics["Baseline"]) / df_metrics["Baseline"]
) * 100

print("Organized Metrics for Visualization:")
display(df_metrics)

In [ ]:
baseline_acc_val = df_metrics.loc[0, "Baseline"]
pruned_acc_val = df_metrics.loc[0, "Pruned"]

baseline_params_val = df_metrics.loc[1, "Baseline"]
pruned_params_val = df_metrics.loc[1, "Pruned"]

baseline_flops_val = df_metrics.loc[2, "Baseline"]
pruned_flops_val = df_metrics.loc[2, "Pruned"]

baseline_lat_val = df_metrics.loc[3, "Baseline"]
pruned_lat_val = df_metrics.loc[3, "Pruned"]

Accuracy vs Model Size Plot

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    baseline_params_val,
    baseline_acc_val,
    color="blue",
    s=120,
    label="Baseline",
    marker="o",
)

plt.scatter(
    pruned_params_val, pruned_acc_val, color="red", s=120, label="Pruned", marker="X"
)

plt.annotate(
    "",
    xy=(pruned_params_val, pruned_acc_val),
    xytext=(baseline_params_val, baseline_acc_val),
    arrowprops=dict(arrowstyle="->", lw=2, color="gray", linestyle="--"),
)

plt.xlabel("Model Parameters (Millions)")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Model Size")

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()

plt.show()

FLOPs vs Latency Plot

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    baseline_flops_val,
    baseline_lat_val,
    color="blue",
    s=120,
    label="Baseline",
    marker="o",
)

plt.scatter(
    pruned_flops_val, pruned_lat_val, color="red", s=120, label="Pruned", marker="X"
)

plt.annotate(
    "",
    xy=(pruned_flops_val, pruned_lat_val),
    xytext=(baseline_flops_val, baseline_lat_val),
    arrowprops=dict(arrowstyle="->", lw=2, color="gray", linestyle="--"),
)

plt.xlabel("FLOPs (G)")
plt.ylabel("Inference Latency (ms)")
plt.title("Latency vs Computational Cost")

plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()

plt.show()

Accuracy vs FLOPs

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(baseline_flops_val, baseline_acc_val, color="blue", s=120, label="Baseline")

plt.scatter(pruned_flops_val, pruned_acc_val, color="red", s=120, label="Pruned")

plt.plot(
    [baseline_flops_val, pruned_flops_val],
    [baseline_acc_val, pruned_acc_val],
    linestyle="--",
    color="gray",
)

plt.xlabel("FLOPs (G)")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Computational Cost")

plt.grid(True, linestyle=":")
plt.legend()

plt.show()

Compression Comparison Bar Chart

In [ ]:
metrics = ["Parameters (M)", "FLOPs (G)", "Latency (ms)"]

baseline_vals = [baseline_params_val, baseline_flops_val, baseline_lat_val]

pruned_vals = [pruned_params_val, pruned_flops_val, pruned_lat_val]

x = range(len(metrics))

plt.figure(figsize=(8, 5))

plt.bar(x, baseline_vals, width=0.4, label="Baseline")
plt.bar([i + 0.4 for i in x], pruned_vals, width=0.4, label="Pruned")

plt.xticks([i + 0.2 for i in x], metrics)

plt.ylabel("Value")
plt.title("Model Compression Comparison")

plt.legend()
plt.show()

Combined Research Figure

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy vs Params
ax1.scatter(
    baseline_params_val, baseline_acc_val, color="blue", s=120, label="Baseline"
)

ax1.scatter(pruned_params_val, pruned_acc_val, color="red", s=120, label="Pruned")

ax1.plot(
    [baseline_params_val, pruned_params_val],
    [baseline_acc_val, pruned_acc_val],
    "--",
    color="gray",
)

ax1.set_xlabel("Parameters (M)")
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy vs Model Size")
ax1.grid(True, linestyle=":")

# FLOPs vs Latency
ax2.scatter(baseline_flops_val, baseline_lat_val, color="blue", s=120, label="Baseline")

ax2.scatter(pruned_flops_val, pruned_lat_val, color="red", s=120, label="Pruned")

ax2.plot(
    [baseline_flops_val, pruned_flops_val],
    [baseline_lat_val, pruned_lat_val],
    "--",
    color="gray",
)

ax2.set_xlabel("FLOPs (G)")
ax2.set_ylabel("Latency (ms)")
ax2.set_title("Latency vs Computational Cost")
ax2.grid(True, linestyle=":")

plt.suptitle("ResNet-18 Structured Pruning Results", fontsize=16)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
fig.savefig("pruning_results.png", dpi=300, bbox_inches="tight")

print("Figure saved as pruning_results.png")

In [ ]:
# Move model back to device if necessary
model = model.to(device)

# Evaluate on full test set
final_test_acc = evaluate(model, testloader)

print("\n--- Final Evaluation ---")

print(f"Pruned Model Test Accuracy: {final_test_acc:.2f}%")

print(f"Baseline Model Test Accuracy: {baseline_acc:.2f}%")

print(f"Net Change: {final_test_acc - baseline_acc:+.2f}%")